# 03 Silver

Lee lo nuevo en bronze con stream de Delta y checkpoint propio. En cada batch
quita duplicados de `(MMSI, BaseDateTime)` y separa filas válidas de cuarentena.
Las válidas entran a silver con un `MERGE` que solo mira los `fecha_dia` de ese
batch. Silver está agrupada con `CLUSTER BY (fecha_dia)` para podar esos días.
Cuarentena es append.

Pasan a silver las filas con MMSI de 9 dígitos, timestamp, latitud, longitud, SOG y COG
en rango. El resto va a cuarentena con `motivo`.

Reglas tomadas del diagnóstico de la Parte 1. `Heading = 511` (centinela, más de la
mitad de la semana) y las dimensiones no eliminan la fila porque no impiden armar una trayectoria.
`SOG = 102.3` es el centinela ITU y cae en `sog_invalido` porque es mayor que 60 nudos. `fecha` solo etiqueta la métrica del job.

No se ejecuta en paralelo con otra sesión.


In [ ]:
%run ./_config


In [ ]:
from pyspark.sql import functions as F

COLUMNAS = [
    "MMSI", "BaseDateTime", "LAT", "LON", "SOG", "COG", "Heading",
    "VesselName", "IMO", "CallSign", "VesselType", "Status",
    "Length", "Width", "Draft", "Cargo", "TransceiverClass",
    "_source_file", "_ingested_at",
]


def con_motivo(df):
    """Marca las reglas fallidas. concat_ws omite los null, asi que '' significa que la fila es valida."""
    mmsi_mal = F.col("MMSI").isNull() | (F.length("MMSI") != 9) | F.col("MMSI").rlike("^0+$")
    ts_mal = F.col("BaseDateTime").isNull()
    lat_mal = F.col("LAT").isNull() | (F.col("LAT") < -90) | (F.col("LAT") > 90) | (F.col("LAT") == 0)
    lon_mal = F.col("LON").isNull() | (F.col("LON") < -180) | (F.col("LON") > 180) | (F.col("LON") == 0)
    sog_mal = F.col("SOG").isNull() | (F.col("SOG") < 0) | (F.col("SOG") > 60)
    cog_mal = F.col("COG").isNull() | (F.col("COG") < 0) | (F.col("COG") > 360)
    motivo = F.concat_ws(
        ",",
        F.when(mmsi_mal, F.lit("mmsi_invalido")),
        F.when(ts_mal, F.lit("sin_timestamp")),
        F.when(lat_mal, F.lit("latitud_invalida")),
        F.when(lon_mal, F.lit("longitud_invalida")),
        F.when(sog_mal, F.lit("sog_invalido")),
        F.when(cog_mal, F.lit("cog_invalido")),
    )
    return df.withColumn("motivo", motivo)


def escribir_silver(sesion, validos) -> None:
    fechas = [
        fila.fecha_dia.isoformat()
        for fila in validos.select("fecha_dia").distinct().collect()
        if fila.fecha_dia is not None
    ]
    if not fechas:
        return
    lista = ", ".join(f"DATE '{dia}'" for dia in fechas)
    vista = "oceanwatch_silver_batch"
    validos.drop("motivo").createOrReplaceTempView(vista)
    sesion.sql(f"""
        MERGE INTO {SILVER} AS destino
        USING {vista} AS origen
        ON destino.fecha_dia IN ({lista})
       AND destino.MMSI = origen.MMSI
       AND destino.BaseDateTime = origen.BaseDateTime
        WHEN NOT MATCHED THEN INSERT *
    """)
    sesion.catalog.dropTempView(vista)


def escribir_cuarentena(invalidos) -> None:
    (
        invalidos.select(*COLUMNAS, "motivo")
        .write.format("delta")
        .mode("append")
        .saveAsTable(CUARENTENA)
    )


def aplicar_silver(batch_df, batch_id: int) -> None:
    sesion = batch_df.sparkSession
    n_entrada = batch_df.count()
    marcado = (
        con_motivo(batch_df.dropDuplicates(["MMSI", "BaseDateTime"]))
        .withColumn("fecha_dia", F.to_date("BaseDateTime"))
    )
    validos = marcado.where(F.col("motivo") == "")
    invalidos = marcado.where(F.col("motivo") != "")
    n_validas = validos.count()
    n_cuarentena = invalidos.count()
    if n_validas:
        escribir_silver(sesion, validos)
    if n_cuarentena:
        escribir_cuarentena(invalidos)
    sesion.sql(f"""
        INSERT INTO {METRICAS}
        SELECT
          DATE '{fecha_dia.isoformat()}',
          'silver',
          CAST(NULL AS BIGINT),
          CAST(NULL AS BIGINT),
          CAST({n_entrada} AS BIGINT),
          CAST({n_validas} AS BIGINT),
          CAST({n_cuarentena} AS BIGINT),
          current_timestamp()
    """)
    print(f"batch {batch_id}: entrada={n_entrada} validas={n_validas} cuarentena={n_cuarentena}")


for consulta in list(spark.streams.active):
    if consulta.name == "oceanwatch_silver_ais":
        consulta.stop()

consulta = (
    spark.readStream.format("delta")
    .table(BRONZE)
    .writeStream.foreachBatch(aplicar_silver)
    .queryName("oceanwatch_silver_ais")
    .option("checkpointLocation", CHECKPOINT_SILVER)
    .trigger(availableNow=True)
    .start()
)
consulta.awaitTermination()
print(f"Ejecución silver {fecha_dia.isoformat()} terminada.")
